# TCCF

Implementação em python do método de recomendação TCCF.
O objetivo principal desta implementação é entender o funcionamento da componente temporal nas recomendações por filtragem colaborativa. Desta forma, a etapa de agrupamento utilizando Cuckoo Search + KMeans será abstraída.

## Dependências

In [1]:
import pandas as pd
import numpy as np

## Time Correlation Coefficient - TCC

$ tcc_{i} = 1-\frac{1}{\sqrt{2\pi\sigma}}(1-exp(-\frac{\Delta t^{2}}{2\sigma^{2}}))$

In [2]:
def compute_tcc(delta_t: float, sigma: float=1.0):
    """
    Calcula o Time Correlation Coefficient (TCC)
    """
    return 1 - (1 / np.sqrt(2 * np.pi * sigma)) * (1 - np.exp(-(delta_t**2) / (2 * sigma**2)))

## Aplicar peso temporal

$r'_u = r_u \times tcc$

In [3]:
def apply_time_weight(df: pd.DataFrame, sigma: float=1.0):
    """
    Aplica o peso temporal aos ratings
    """
    df = df.copy()

    latest_time = df['timestamp'].max()
    df["delta_t"] = (latest_time - df["timestamp"])

    df["tcc"] = df["delta_t"].apply(lambda x: compute_tcc(x, sigma))

    df["rating_tcc"] = df["rating"] * df["tcc"]

    return df

## Calcular a similaridade entre 2 usuários
Utiliza como base a matriz de notas

$sim(u, v)=\frac{\sum_{i\epsilon P_{uv}}^{}(r'_{ui} - \bar{r'_{u}})(r'_{vi} - \bar{r'_{vu}})}{\sqrt{\sum_{i\epsilon P_{uv}}^{}(r'_{ui} - \bar{r'_{u}})^2}\sqrt{\sum_{i\epsilon P_{uv}}^{}(r'_{vi} - \bar{r'_{v}})^2}}$

$r'$ são as notas ponderadas pelo $TCC$

In [4]:
def similarity(user1: str, user2: str, ratings_matrix: pd.DataFrame):
    """
    Calcula similaridade entre dois usuários
    """

    # i E P_{uv}
    common_items = ratings_matrix.loc[user1].dropna().index.intersection(
        ratings_matrix.loc[user2].dropna().index
    )

    if len(common_items) == 0:
        return 0

    r1 = ratings_matrix.loc[user1, common_items]
    r2 = ratings_matrix.loc[user2, common_items]

    mean1 = r1.mean()
    mean2 = r2.mean()

    numerator = np.sum((r1 - mean1) * (r2 - mean2))
    denominator = np.sqrt(np.sum((r1 - mean1) ** 2)) * np.sqrt(np.sum((r2 - mean2) ** 2))

    if denominator == 0:
        return 0

    return numerator / denominator

## Predição das respostas
Utiliza a similaridade entre os usuários para projetar a nota para o item.

$P_{ui} = \bar{r_u}+\frac{\sum_{v \epsilon NBS_u}^{}sim(u, v) \times (r_{vi} - \bar{r_v})}{\sum_{v \epsilon NBS_u}^{}sim(u, v)}$

In [5]:
def predict_rating(user, item, ratings_matrix, similarities):
    """
    Prediz rating de user para item
    """

    neighbors = similarities.keys()

    numerator = 0
    denominator = 0

    for v in neighbors:

        if item not in ratings_matrix.columns:
            continue

        sim = similarities[v]

        r_vi = ratings_matrix.loc[v, item]
        mean_v = ratings_matrix.loc[v].mean()

        if np.isnan(r_vi):
            continue

        numerator += sim * (r_vi - mean_v)
        denominator += abs(sim)

    if denominator == 0:
        return np.nan

    mean_u = ratings_matrix.loc[user].mean()

    return mean_u + numerator / denominator

## Popularidade do item (cold start)

$ItemPop_i = \frac{\left | U_i\right |}{\sqrt{\sum_{i \epsilon I}\left|U_i\right|^2}}$

$U_i$ é a quantidade de usuários que consumiram o item $i$

In [6]:
def popularity_recommendation(df, top_n=10):
    """
    Recomendação baseada em popularidade
    """

    popularity = df.groupby("item_id")["user_id"].count()

    pop_score = popularity / np.sqrt(np.sum(popularity ** 2))

    return pop_score.sort_values(ascending=False).head(top_n).index.tolist()

## Função de recomendação

In [7]:
def recommend_items(user, ratings_matrix, similarities, top_n=10):

    items = ratings_matrix.columns
    predictions = {}

    for item in items:

        if not np.isnan(ratings_matrix.loc[user, item]):
            continue

        pred = predict_rating(user, item, ratings_matrix, similarities)

        if not np.isnan(pred):
            predictions[item] = pred

    ranked = sorted(predictions.items(), key=lambda x: x[1], reverse=True)

    return [i for i in ranked[:top_n]]

## Pipeline completo

In [8]:
def tccf_recommend(df, user_id, cluster_id, sigma=1.0, top_n=10):

    # filtra cluster
    cluster_df = df[df["cluster"] == cluster_id]

    # aplica peso temporal
    cluster_df = apply_time_weight(cluster_df, sigma)

    # matriz user-item
    ratings_matrix = cluster_df.pivot_table(
        index="user_id",
        columns="item_id",
        values="rating_tcc"
    )

    # usuário novo, sem histórico
    if user_id not in ratings_matrix.index:
        return popularity_recommendation(cluster_df, top_n)

    # calcula similaridades
    similarities = {}

    for other_user in ratings_matrix.index:

        if other_user == user_id:
            continue

        sim = similarity(user_id, other_user, ratings_matrix)

        if sim > 0:
            similarities[other_user] = sim

    return recommend_items(user_id, ratings_matrix, similarities, top_n)

## Testes

In [13]:
data = [

    ("U2","M1",4,1,1),
    ("U2","M2",5,2,1),
    ("U2","M3",4,3,1),
    ("U2","M8",4,4,1),

    ("U4","M1",4,51,1),
    ("U4","M2",5,52,1),
    ("U4","M3",4,53,1),
    ("U4","M6",4,54,1),

    # Usuário alvo
    ("U_target","M1",5,55,1),
    ("U_target","M2",5,56,1),
    ("U_target","M3",4,57,1),

]

df = pd.DataFrame(
    data,
    columns=["user_id","item_id","rating","timestamp","cluster"]
)

In [14]:
df.pivot_table(
    index="user_id",
    columns="item_id",
    values="rating"
)

item_id,M1,M2,M3,M6,M8
user_id,,,,,
U2,4.0,5.0,4.0,NaN,4.0
U4,4.0,5.0,4.0,4.0,NaN
U_target,5.0,5.0,4.0,NaN,NaN


In [17]:
print("====== Considerando variável temporal ======")
rec = tccf_recommend(df, "U_target", 1, sigma=5, top_n=4)

print("Recomendações")
for rec_ in rec:
    print(f"Filme: {rec_[0]} - Nota: {rec_[1]}")


print("====== Desconsiderando variável temporal ======")
df_padrao = df.copy()
df_padrao["timestamp"] = 1

rec = tccf_recommend(df_padrao, "U_target", 1, sigma=5, top_n=4)

print("Recomendações")
for rec_ in rec:
    print(f"Filme: {rec_[0]} - Nota: {rec_[1]}")


====== Considerando variável temporal ======
Recomendações
Filme: M6 - Nota: 4.5279254284807084
Filme: M8 - Nota: 4.432520102064691
====== Desconsiderando variável temporal ======
Recomendações
Filme: M6 - Nota: 4.416666666666667
Filme: M8 - Nota: 4.416666666666667
